In [1]:
import numpy as np
import pandas as pd
import joblib

print("Fusion logic notebook ready.")

Fusion logic notebook ready.


In [2]:
class FusionEngine:
    def __init__(self, face_consecutive_frames=3, fatigue_consecutive_frames=3, ear_threshold=0.20):
        self.face_negative_streak = 0
        self.fatigue_low_ear_streak = 0
        self.behavioral_flag = False

        self.face_consecutive_frames = face_consecutive_frames
        self.fatigue_consecutive_frames = fatigue_consecutive_frames
        self.ear_threshold = ear_threshold
        self.negative_emotions = {'frustrated', 'sad', 'fear'}

    def update_face_frame(self, emotion_label):
        if emotion_label in self.negative_emotions:
            self.face_negative_streak += 1
        else:
            self.face_negative_streak = 0
        return self.face_negative_streak >= self.face_consecutive_frames

    def update_fatigue_frame(self, avg_ear):
        if avg_ear is None:
            return False
        if avg_ear < self.ear_threshold:
            self.fatigue_low_ear_streak += 1
        else:
            self.fatigue_low_ear_streak = 0
        return self.fatigue_low_ear_streak >= self.fatigue_consecutive_frames

    def check_behavioral(self, isolation_forest_prediction):
        self.behavioral_flag = (isolation_forest_prediction == 'struggling')
        return self.behavioral_flag

    def decide_action(self, face_triggered, fatigue_triggered):
        if fatigue_triggered:
            return 'BREAK_SUGGESTION_FATIGUE', "Player appears fatigued - suggest a short break + bonus life"
        elif face_triggered and self.behavioral_flag:
            return 'BIT_MENU_URGENT', "Frustrated AND struggling - show Bit's menu, emphasize 'Review Basics'"
        elif face_triggered:
            return 'BREAK_SUGGESTION_EMOTION', "Player appears frustrated - suggest a short break"
        elif self.behavioral_flag:
            return 'BIT_MENU', "Struggling with content - show Bit's menu (Easier/Review Basics/Continue)"
        else:
            return 'NONE', "Continue normal gameplay"


print("FusionEngine class defined.")

FusionEngine class defined.


In [3]:
iso_forest_loaded = joblib.load('../models/isolation_forest_model.joblib')
scaler_loaded = joblib.load('../models/behavioral_scaler.joblib')

print("Isolation Forest model loaded for fusion pipeline.")

def predict_struggle(attempts, time_taken, misconception_repeats, combo_breaks):
    features = pd.DataFrame([{
        'attempts_count': attempts,
        'time_taken_seconds': time_taken,
        'misconception_repeat_count': misconception_repeats,
        'combo_breaks': combo_breaks
    }])
    features_scaled = scaler_loaded.transform(features)
    prediction_raw = iso_forest_loaded.predict(features_scaled)
    return 'struggling' if prediction_raw[0] == -1 else 'typical'


test_prediction = predict_struggle(attempts=9, time_taken=95, misconception_repeats=3, combo_breaks=2)
print(f"Test round prediction: {test_prediction}")


test_prediction_2 = predict_struggle(attempts=2, time_taken=35, misconception_repeats=0, combo_breaks=0)
print(f"Test round prediction: {test_prediction_2}")

Isolation Forest model loaded for fusion pipeline.
Test round prediction: struggling
Test round prediction: typical


In [6]:
engine = FusionEngine()

print("=== SIMULATED GAMEPLAY SESSION (Level example) ===\n")

simulated_frames = [
    {'emotion': 'neutral',     'ear': 0.34},
    {'emotion': 'neutral',     'ear': 0.32},
    {'emotion': 'frustrated',  'ear': 0.30},
    {'emotion': 'frustrated',  'ear': 0.29},
    {'emotion': 'frustrated',  'ear': 0.28},
    {'emotion': 'sad',         'ear': 0.27},
]

for i, frame in enumerate(simulated_frames):
    face_triggered = engine.update_face_frame(frame['emotion'])
    fatigue_triggered = engine.update_fatigue_frame(frame['ear'])
    action, message = engine.decide_action(face_triggered, fatigue_triggered)
    print(f"Frame {i+1}: emotion={frame['emotion']:10s} ear={frame['ear']:.2f} "
          f"| face_streak={engine.face_negative_streak} fatigue_streak={engine.fatigue_low_ear_streak} "
          f"-> Action: {action}")

print("\n--- Rounds 1-3 ivara una gaman, Behavioral (Isolation Forest) check ekak karanawa ---")

round_prediction = predict_struggle(attempts=15, time_taken=140, misconception_repeats=5, combo_breaks=4)
behavioral_triggered = engine.check_behavioral(round_prediction)
print(f"Behavioral prediction: {round_prediction} -> Flag set: {behavioral_triggered}")

print("\n--- Behavioral flag set unaata passe, ithuru frames tika process karamu (player thawath frustrated wenawa) ---")

more_frames = [
    {'emotion': 'frustrated', 'ear': 0.31},
    {'emotion': 'frustrated', 'ear': 0.29},
]

for i, frame in enumerate(more_frames):
    face_triggered = engine.update_face_frame(frame['emotion'])
    fatigue_triggered = engine.update_fatigue_frame(frame['ear'])
    action, message = engine.decide_action(face_triggered, fatigue_triggered)
    print(f"Frame {i+7}: emotion={frame['emotion']:10s} ear={frame['ear']:.2f} "
          f"-> Action: {action} | {message}")

=== SIMULATED GAMEPLAY SESSION (Level example) ===

Frame 1: emotion=neutral    ear=0.34 | face_streak=0 fatigue_streak=0 -> Action: NONE
Frame 2: emotion=neutral    ear=0.32 | face_streak=0 fatigue_streak=0 -> Action: NONE
Frame 3: emotion=frustrated ear=0.30 | face_streak=1 fatigue_streak=0 -> Action: NONE
Frame 4: emotion=frustrated ear=0.29 | face_streak=2 fatigue_streak=0 -> Action: NONE
Frame 5: emotion=frustrated ear=0.28 | face_streak=3 fatigue_streak=0 -> Action: BREAK_SUGGESTION_EMOTION
Frame 6: emotion=sad        ear=0.27 | face_streak=4 fatigue_streak=0 -> Action: BREAK_SUGGESTION_EMOTION

--- Rounds 1-3 ivara una gaman, Behavioral (Isolation Forest) check ekak karanawa ---
Behavioral prediction: struggling -> Flag set: True

--- Behavioral flag set unaata passe, ithuru frames tika process karamu (player thawath frustrated wenawa) ---
Frame 7: emotion=frustrated ear=0.31 -> Action: BIT_MENU_URGENT | Frustrated AND struggling - show Bit's menu, emphasize 'Review Basics'
Fram